In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
from dbfread import DBF

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "Data"
    / "raw"
    / "Himalayan Database"
    / "HIMDATA"
)

# Load source tables
peaks = pd.DataFrame(
    iter(DBF(DATA_PATH / "peaks.DBF", load=True))
)

exped = pd.DataFrame(
    iter(DBF(DATA_PATH / "exped.DBF", load=True))
)

print("Peaks:", peaks.shape)
print("Expeditions:", exped.shape)

Peaks: (503, 25)
Expeditions: (11854, 66)


In [3]:
peak_info = peaks[
    [
        "PEAKID",
        "PKNAME",
        "HEIGHTM",
        "LOCATION",
        "REGION",
        "OPEN",
        "TREKKING"
    ]
].copy()

expedition_data = exped.merge(
    peak_info,
    on="PEAKID",
    how="left"
)

print("Expedition data shape:", expedition_data.shape)

Expedition data shape: (11854, 72)


In [4]:
expedition_data.dtypes

EXPID         str
PEAKID        str
YEAR          str
SEASON      int64
HOST        int64
            ...  
HEIGHTM     int64
LOCATION      str
REGION      int64
OPEN         bool
TREKKING     bool
Length: 72, dtype: object

In [5]:
success_cols = [
    "SUCCESS1",
    "SUCCESS2",
    "SUCCESS3",
    "SUCCESS4"
]

expedition_data["EXPEDITION_SUCCESS"] = (
    expedition_data[success_cols].any(axis=1)
)

print(
    expedition_data["EXPEDITION_SUCCESS"].value_counts()
)

EXPEDITION_SUCCESS
True     6684
False    5170
Name: count, dtype: int64


In [6]:
feature_cols = [
    "YEAR",
    "SEASON",
    "PEAKID",
    "HEIGHTM",
    "LOCATION",
    "REGION",
    "OPEN",
    "TREKKING",
    "TOTMEMBERS",
    "CAMPS",
    "O2USED",
    "O2CLIMB",
    "O2DESCENT",
    "O2SLEEP",
    "O2MEDICAL"
]

X = expedition_data[feature_cols].copy()
y = expedition_data["EXPEDITION_SUCCESS"].copy()

print("Features:", X.shape)
print("Target:", y.shape)

X.head()

Features: (11854, 15)
Target: (11854,)


,YEAR,SEASON,PEAKID,HEIGHTM,LOCATION,REGION,OPEN,TREKKING,TOTMEMBERS,CAMPS,O2USED,O2CLIMB,O2DESCENT,O2SLEEP,O2MEDICAL
0,1960,1,ANN2,7937,Annapurna Himal,5,True,False,10,6,True,True,False,True,False
1,1969,3,ANN2,7937,Annapurna Himal,5,True,False,10,6,False,False,False,False,False
2,1973,1,ANN2,7937,Annapurna Himal,5,True,False,6,5,False,False,False,False,False
3,1978,3,ANN2,7937,Annapurna Himal,5,True,False,2,0,False,False,False,False,False
4,1979,3,ANN2,7937,Annapurna Himal,5,True,False,3,0,False,False,False,False,False


In [7]:
feature_check = pd.DataFrame({
    "dtype": X.dtypes,
    "missing": X.isnull().sum(),
    "unique_values": X.nunique()
})

feature_check

,dtype,missing,unique_values
YEAR,str,0,98
SEASON,int64,0,5
PEAKID,str,0,439
HEIGHTM,int64,0,391
LOCATION,str,0,375
REGION,int64,0,7
OPEN,bool,0,2
TREKKING,bool,0,2
TOTMEMBERS,int64,0,53
CAMPS,int64,0,17


In [8]:
X["YEAR"] = pd.to_numeric(X["YEAR"], errors="coerce")

boolean_cols = [
    "OPEN",
    "TREKKING",
    "O2USED",
    "O2CLIMB",
    "O2DESCENT",
    "O2SLEEP",
    "O2MEDICAL"
]

X[boolean_cols] = X[boolean_cols].astype(int)

print(X.dtypes)

YEAR          int64
SEASON        int64
PEAKID          str
HEIGHTM       int64
LOCATION        str
REGION        int64
OPEN          int64
TREKKING      int64
TOTMEMBERS    int64
CAMPS         int64
O2USED        int64
O2CLIMB       int64
O2DESCENT     int64
O2SLEEP       int64
O2MEDICAL     int64
dtype: object


In [9]:
X_encoded = pd.get_dummies(
    X,
    columns=["PEAKID", "LOCATION"],
    drop_first=True,
    dtype=int
)

print("Original features:", X.shape)
print("Encoded features:", X_encoded.shape)

Original features: (11854, 15)
Encoded features: (11854, 825)


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (9483, 825)
X_test: (2371, 825)
y_train: (9483,)
y_test: (2371,)


In [11]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


c:\Projects\PeakRisk\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
y_pred = model.predict(X_test)

print(y_pred[:20])

[ True  True False False  True  True False False False False  True False
  True False False  True  True  True  True False]


In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=3000,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


c:\Projects\PeakRisk\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
y_pred = model.predict(X_test)

print(y_pred[:20])

[ True  True False False  True False  True False False False  True  True
  True False False  True  True  True  True False]


In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.7305
Precision: 0.75
Recall   : 0.7831
F1 Score : 0.7662

Classification Report:
              precision    recall  f1-score   support

       False       0.70      0.66      0.68      1034
        True       0.75      0.78      0.77      1337

    accuracy                           0.73      2371
   macro avg       0.73      0.72      0.72      2371
weighted avg       0.73      0.73      0.73      2371

Confusion Matrix:
[[ 685  349]
 [ 290 1047]]


In [16]:
processed_path = PROJECT_ROOT / "Data" / "processed"

processed_path.mkdir(parents=True, exist_ok=True)

X_encoded.to_csv(
    processed_path / "peakrisk_features.csv",
    index=False
)

y.to_csv(
    processed_path / "peakrisk_target.csv",
    index=False
)

print("Feature dataset saved.")
print("Target dataset saved.")

Feature dataset saved.
Target dataset saved.


In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")

rf_pred = rf_model.predict(X_test)

print("Predictions generated.")

Random Forest training completed.
Predictions generated.


In [18]:
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

Random Forest Results
---------------------
Accuracy : 0.7436
Precision: 0.7674
Recall   : 0.7823
F1 Score : 0.7748

Classification Report:
              precision    recall  f1-score   support

       False       0.71      0.69      0.70      1034
        True       0.77      0.78      0.77      1337

    accuracy                           0.74      2371
   macro avg       0.74      0.74      0.74      2371
weighted avg       0.74      0.74      0.74      2371

Confusion Matrix:
[[ 717  317]
 [ 291 1046]]


In [19]:
from sklearn.model_selection import GroupShuffleSplit

groups = expedition_data["PEAKID"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X_encoded, y, groups=groups)
)

X_train_group = X_encoded.iloc[train_idx]
X_test_group = X_encoded.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

print("Training:", X_train_group.shape)
print("Testing :", X_test_group.shape)

print(
    "Unique training mountains:",
    expedition_data.iloc[train_idx]["PEAKID"].nunique()
)

print(
    "Unique testing mountains:",
    expedition_data.iloc[test_idx]["PEAKID"].nunique()
)

Training: (10740, 825)
Testing : (1114, 825)
Unique training mountains: 351
Unique testing mountains: 88


In [20]:
model_features = [
    "YEAR",
    "SEASON",
    "HEIGHTM",
    "REGION",
    "OPEN",
    "TREKKING",
    "TOTMEMBERS",
    "CAMPS",
    "O2USED",
    "O2CLIMB",
    "O2DESCENT",
    "O2SLEEP",
    "O2MEDICAL"
]

X_model = X[model_features].copy()

print("Model features:", X_model.shape)

Model features: (11854, 13)


In [21]:
from sklearn.model_selection import GroupShuffleSplit

groups = expedition_data["PEAKID"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X_model, y, groups=groups)
)

X_train_group = X_model.iloc[train_idx]
X_test_group = X_model.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

print("Training:", X_train_group.shape)
print("Testing :", X_test_group.shape)

print(
    "Unique training mountains:",
    expedition_data.iloc[train_idx]["PEAKID"].nunique()
)

print(
    "Unique testing mountains:",
    expedition_data.iloc[test_idx]["PEAKID"].nunique()
)

Training: (10740, 13)
Testing : (1114, 13)
Unique training mountains: 351
Unique testing mountains: 88


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

rf_group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_group_model.fit(X_train_group, y_train_group)

rf_group_pred = rf_group_model.predict(X_test_group)

print("Accuracy :", accuracy_score(y_test_group, rf_group_pred))
print("Precision:", precision_score(y_test_group, rf_group_pred))
print("Recall   :", recall_score(y_test_group, rf_group_pred))
print("F1 Score :", f1_score(y_test_group, rf_group_pred))

print("\nClassification Report:")
print(classification_report(y_test_group, rf_group_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_group, rf_group_pred))

Accuracy : 0.6606822262118492
Precision: 0.6283924843423799
Recall   : 0.6007984031936128
F1 Score : 0.6142857142857143

Classification Report:
              precision    recall  f1-score   support

       False       0.69      0.71      0.70       613
        True       0.63      0.60      0.61       501

    accuracy                           0.66      1114
   macro avg       0.66      0.66      0.66      1114
weighted avg       0.66      0.66      0.66      1114


Confusion Matrix:
[[435 178]
 [200 301]]


In [23]:
feature_importance = pd.DataFrame({
    "Feature": X_model.columns,
    "Importance": rf_group_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

feature_importance

,Feature,Importance
0,YEAR,0.299394
2,HEIGHTM,0.198259
6,TOTMEMBERS,0.162144
7,CAMPS,0.120403
9,O2CLIMB,0.057872
8,O2USED,0.044127
3,REGION,0.043839
1,SEASON,0.043117
11,O2SLEEP,0.014982
4,OPEN,0.005499


In [24]:
pre_expedition_features = [
    "YEAR",
    "SEASON",
    "HEIGHTM",
    "REGION",
    "OPEN",
    "TREKKING",
    "TOTMEMBERS",
    "CAMPS"
]

X_pre = X[pre_expedition_features].copy()

print("Pre-expedition features:", X_pre.shape)
print(X_pre.columns.tolist())

Pre-expedition features: (11854, 8)
['YEAR', 'SEASON', 'HEIGHTM', 'REGION', 'OPEN', 'TREKKING', 'TOTMEMBERS', 'CAMPS']


In [25]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Grouped split by mountain
groups = expedition_data["PEAKID"]

gss_pre = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_pre, test_idx_pre = next(
    gss_pre.split(X_pre, y, groups=groups)
)

X_train_pre = X_pre.iloc[train_idx_pre]
X_test_pre = X_pre.iloc[test_idx_pre]

y_train_pre = y.iloc[train_idx_pre]
y_test_pre = y.iloc[test_idx_pre]

print("Training:", X_train_pre.shape)
print("Testing :", X_test_pre.shape)

print(
    "Unique training mountains:",
    expedition_data.iloc[train_idx_pre]["PEAKID"].nunique()
)

print(
    "Unique testing mountains:",
    expedition_data.iloc[test_idx_pre]["PEAKID"].nunique()
)

Training: (10740, 8)
Testing : (1114, 8)
Unique training mountains: 351
Unique testing mountains: 88


In [26]:
rf_pre_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_pre_model.fit(X_train_pre, y_train_pre)

rf_pre_pred = rf_pre_model.predict(X_test_pre)

print("Accuracy :", accuracy_score(y_test_pre, rf_pre_pred))
print("Precision:", precision_score(y_test_pre, rf_pre_pred))
print("Recall   :", recall_score(y_test_pre, rf_pre_pred))
print("F1 Score :", f1_score(y_test_pre, rf_pre_pred))

print("\nClassification Report:")
print(classification_report(y_test_pre, rf_pre_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_pre, rf_pre_pred))

Accuracy : 0.644524236983842
Precision: 0.6003824091778203
Recall   : 0.626746506986028
F1 Score : 0.61328125

Classification Report:
              precision    recall  f1-score   support

       False       0.68      0.66      0.67       613
        True       0.60      0.63      0.61       501

    accuracy                           0.64      1114
   macro avg       0.64      0.64      0.64      1114
weighted avg       0.65      0.64      0.65      1114


Confusion Matrix:
[[404 209]
 [187 314]]


In [27]:
pre_feature_importance = pd.DataFrame({
    "Feature": X_pre.columns,
    "Importance": rf_pre_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

pre_feature_importance

,Feature,Importance
0,YEAR,0.357789
6,TOTMEMBERS,0.186043
2,HEIGHTM,0.179904
7,CAMPS,0.170115
1,SEASON,0.050831
3,REGION,0.047858
4,OPEN,0.005448
5,TREKKING,0.002012


In [28]:
baseline_accuracy = y_test_pre.value_counts(normalize=True).max()

model_accuracy = accuracy_score(
    y_test_pre,
    rf_pre_pred
)

print("Baseline Accuracy:", baseline_accuracy)
print("Model Accuracy   :", model_accuracy)
print(
    "Improvement      :",
    model_accuracy - baseline_accuracy
)

Baseline Accuracy: 0.5502692998204668
Model Accuracy   : 0.644524236983842
Improvement      : 0.09425493716337519


In [29]:
model_comparison = pd.DataFrame({
    "Model": [
        "Majority Baseline",
        "Logistic Regression - Random Split",
        "Random Forest - Random Split",
        "Random Forest - Grouped Split (13 Features)",
        "Random Forest - Pre-Expedition (8 Features)"
    ],
    "Accuracy": [
        baseline_accuracy,
        0.7305,
        0.7436,
        0.6606822262,
        0.6445242370
    ],
    "Precision": [
        None,
        0.7500,
        0.7674,
        0.6283924843,
        0.6003824092
    ],
    "Recall": [
        None,
        0.7831,
        0.7823,
        0.6007984032,
        0.6267465070
    ],
    "F1 Score": [
        None,
        0.7662,
        0.7748,
        0.6142857143,
        0.6132812500
    ]
})

model_comparison

model_comparison.round(4)

,Model,Accuracy,Precision,Recall,F1 Score
0,Majority Baseline,0.5503,NaN,NaN,NaN
1,Logistic Regression - Random Split,0.7305,0.7500,0.7831,0.7662
2,Random Forest - Random Split,0.7436,0.7674,0.7823,0.7748
3,Random Forest - Grouped Split (13 Features),0.6607,0.6284,0.6008,0.6143
4,Random Forest - Pre-Expedition (8 Features),0.6445,0.6004,0.6267,0.6133
